# Where Will a Gym Thrive in Utah? — Demo

**STAT 486 Final Project**

This notebook demonstrates the full pipeline end-to-end using pre-collected data.  
Run cells top-to-bottom — no API keys required.

**Research Question:** Can neighborhood socioeconomic characteristics and business category predict whether a fitness business will thrive in Utah?

**Thriving** = open + Yelp rating ≥ 4.0 + review_count ≥ 10

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, r2_score, mean_squared_error, silhouette_score

sns.set_style('whitegrid')
print('Libraries loaded.')

## 1. Dataset Overview

Data collected via Yelp Fusion API + U.S. Census ACS 2022 (5-year estimates).

In [ ]:
# Load pre-collected data from thriving_model/data/
df = pd.read_csv('thriving_model/data/utah_fitness_v2.csv', dtype={'zip_code': str})
clf_df = pd.read_csv('thriving_model/data/features_classifier.csv', dtype={'zip_code': str})
reg_df = pd.read_csv('thriving_model/data/features_regressor.csv', dtype={'zip_code': str})

print(f'Full dataset:      {len(df):,} businesses across {df["zip_code"].nunique()} zip codes')
print(f'Classifier subset: {len(clf_df):,} businesses ({clf_df["is_thriving"].sum():.0f} thriving / {(~clf_df["is_thriving"].astype(bool)).sum()} not)')
print(f'Regressor subset:  {len(reg_df):,} open businesses')
print(f'\nTarget distribution:')
print(clf_df['is_thriving'].value_counts().rename({0: 'Not Thriving', 1: 'Thriving'}))

df.head(3)

In [ ]:
# Quick EDA: category distribution and thriving rates
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category counts
cat_counts = clf_df['category_group'].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, color='steelblue')
axes[0].set_title('Business Count by Category Group')
axes[0].set_xlabel('Category'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Thriving rate by category
thriving_rate = clf_df.groupby('category_group')['is_thriving'].mean().sort_values(ascending=False)
axes[1].bar(thriving_rate.index, thriving_rate.values * 100, color='seagreen')
axes[1].set_title('Thriving Rate by Category Group (%)')
axes[1].set_xlabel('Category'); axes[1].set_ylabel('% Thriving')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(clf_df['is_thriving'].mean() * 100, color='red', linestyle='--', label='Overall avg')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Overall thriving rate: {clf_df["is_thriving"].mean():.1%}')

In [ ]:
# Income vs. competition scatter
fig, ax = plt.subplots(figsize=(8, 5))
colors = clf_df['is_thriving'].map({1: 'seagreen', 0: 'tomato'})
ax.scatter(clf_df['income_per_competitor'], clf_df['success_score'],
           c=colors, alpha=0.6, edgecolors='white', linewidths=0.5)
ax.set_xlabel('Income per Competitor (log scale would help)')
ax.set_ylabel('Success Score')
ax.set_title('Income per Competitor vs Success Score')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='seagreen', label='Thriving'), Patch(color='tomato', label='Not Thriving')])
plt.tight_layout()
plt.show()

## 2. Task A: Binary Classifier — Will this business thrive?

Pipeline: median imputation → standard scaling → one-hot encoding → model  
Tuned with 5-fold GridSearchCV.

In [ ]:
NUMERIC_FEATURES = [
    'median_income', 'median_home_value', 'median_age',
    'pct_bachelors', 'pct_prime_gym_age', 'total_pop',
    'competition_1km', 'competition_3km',
    'market_gap', 'gym_density_per_1k', 'income_per_competitor',
    'price',
]
CATEGORICAL_FEATURES = ['category_group']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), CATEGORICAL_FEATURES),
])

X = clf_df[ALL_FEATURES]
y = clf_df['is_thriving'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)} | Positive rate: {y_train.mean():.2f}')

In [ ]:
# Random Forest Classifier (best performer from full grid search)
rf_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_split=2,
        class_weight='balanced', random_state=42
    ))
])
rf_pipe.fit(X_train, y_train)

y_pred = rf_pipe.predict(X_test)
y_prob = rf_pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)

print(f'Random Forest — AUC: {auc:.3f} | Accuracy: {acc:.3f}')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', label=f'Random Forest (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', label='Random baseline')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Is This Business Thriving?')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances
ohe_names = (
    rf_pipe.named_steps['pre']
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(CATEGORICAL_FEATURES)
).tolist()
all_names = NUMERIC_FEATURES + ohe_names
importances = rf_pipe.named_steps['model'].feature_importances_

fi_df = pd.DataFrame({'feature': all_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='steelblue')
ax.set_title('Top 12 Feature Importances — Random Forest Classifier')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
print(fi_df.head(5).to_string(index=False))

## 3. Task B: Regressor — How successful will an open business be?

Target: `success_score` = 0.5 × norm(rating) + 0.5 × norm(log_reviews)

In [ ]:
X_reg = reg_df[ALL_FEATURES]
y_reg = reg_df['success_score']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Gradient Boosting Regressor (best performer)
gb_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    ))
])
gb_pipe.fit(X_train_r, y_train_r)

y_pred_r = gb_pipe.predict(X_test_r)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
r2 = r2_score(y_test_r, y_pred_r)
print(f'Gradient Boosting — R²: {r2:.3f} | RMSE: {rmse:.3f}')

# Predicted vs actual
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test_r, y_pred_r, alpha=0.4, color='tomato', edgecolors='white', linewidths=0.3)
lims = [min(y_test_r.min(), y_pred_r.min()), max(y_test_r.max(), y_pred_r.max())]
ax.plot(lims, lims, 'k--', label='Perfect prediction')
ax.set_xlabel('Actual Success Score'); ax.set_ylabel('Predicted Success Score')
ax.set_title(f'Gradient Boosting Regressor (R²={r2:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Task C: K-Means Market Segmentation

Cluster zip codes into market archetypes based on demographics, competition, and performance.

In [ ]:
zip_features = df.groupby('zip_code').agg(
    median_income=('median_income', 'first'),
    median_home_value=('median_home_value', 'first'),
    pct_prime_gym_age=('pct_prime_gym_age', 'first'),
    pct_bachelors=('pct_bachelors', 'first'),
    total_pop=('total_pop', 'first'),
    gyms_in_zip=('gyms_in_zip', 'first'),
    market_gap=('market_gap', 'first'),
    avg_rating=('rating', lambda x: x[x > 0].mean()),
    lat=('latitude', 'mean'),
    lon=('longitude', 'mean'),
).dropna().reset_index()

CLUSTER_FEATURES = [
    'median_income', 'median_home_value', 'pct_prime_gym_age',
    'pct_bachelors', 'total_pop', 'gyms_in_zip', 'market_gap', 'avg_rating',
]
scaler = StandardScaler()
X_clust = scaler.fit_transform(zip_features[CLUSTER_FEATURES].fillna(0))
print(f'Clustering {len(zip_features)} zip codes on {len(CLUSTER_FEATURES)} features')

# Silhouette scores
K_range = range(2, 9)
sil_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust)
    sil_scores.append(silhouette_score(X_clust, labels))

best_k = list(K_range)[np.argmax(sil_scores)]
print(f'Best K by silhouette: {best_k} (score={max(sil_scores):.3f})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), sil_scores, 'o-', color='tomato')
ax.axvline(best_k, color='gray', linestyle='--', label=f'Best K={best_k}')
ax.set_xlabel('K'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score by K')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit final K-Means
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
zip_features['cluster'] = km_final.fit_predict(X_clust)

# Geographic map
cmap = plt.cm.get_cmap('tab10', best_k)
fig, ax = plt.subplots(figsize=(8, 10))
for cid in sorted(zip_features['cluster'].unique()):
    sub = zip_features[zip_features['cluster'] == cid]
    ax.scatter(sub['lon'], sub['lat'],
               c=[cmap(cid)], label=f'Cluster {cid}',
               s=sub['total_pop'] / 500, alpha=0.7,
               edgecolors='white', linewidths=0.5)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Utah Fitness Market Segments (K={best_k})\npoint size = population')
ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Thriving rate per cluster
merged = df.merge(zip_features[['zip_code', 'cluster']], on='zip_code', how='left')
thriving_by_cluster = (
    merged[merged['is_thriving'].notna()]
    .groupby('cluster')['is_thriving']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'pct_thriving', 'count': 'n_businesses'})
    .round(3)
    .sort_values('pct_thriving', ascending=False)
)
print('Thriving rate by cluster:')
print(thriving_by_cluster)

# Cluster profiles
print('\nCluster mean profiles (selected features):')
print(zip_features.groupby('cluster')[['median_income', 'pct_prime_gym_age', 'gyms_in_zip', 'avg_rating']].mean().round(1))

## 5. Summary

| Task | Best Model | Key Metric |
|---|---|---|
| Binary classifier | Random Forest | AUC = 0.927, Accuracy = 87.9% |
| Regressor | Gradient Boosting | R² = 0.244, RMSE = 0.263 |
| Clustering | K-Means K=8 | Thriving rates range 33%–100% across clusters |

**Key findings:**
1. **Business category is the strongest predictor** — general gyms thrive at the highest rates
2. **Income per competitor is the best location signal** — wealthier areas with fewer gyms are ideal
3. **Competition within 3km** is the top regressor feature — market saturation hurts success scores
4. **K-Means reveals clear market archetypes** — affluent suburban segments have dramatically higher thriving rates than dense urban cores

**Limitations:** Yelp API does not return closed businesses, so we cannot model failure directly. Business age/longevity data was not obtainable at scale. R² of 0.244 suggests important unmeasured factors exist.